# 4 · A picture chosen per tile

Extrapolation's one learned piece worth keeping was a decision (its notebook 13): per tile, the
search's N+1 or N, by a 400-weight network on six numbers of the search's own costs - +0.18 dB in
the graph for 7% more time. A decision cannot hedge: it picks one sharp picture, never a blend of
them, which is what every network that drew pixels or motion learned to do.

Interpolation has more to pick from. Every picture `models.Decided` makes is N read at -s·d and
N+2 at +s·d, mixed a : 1 - a, d the search's offset (notebook 3's reach):

| choice | s | a | |
| --- | --- | --- | --- |
| search | 1 | ½ | the published graph |
| mean | 0 | ½ | the mock |
| hold N, hold N+2 | 0 | 1, 0 | either frame shown again |
| N moved, N+2 moved | 1 | 1, 0 | either frame moved half way alone |

The last four draw a change that happened in one display frame once - a frame early or late -
rather than twice at half strength, as notebook 2's `Commit` does per pixel by a fixed rule. A
network picks per tile from the six numbers `Decided.features` measures:

- **two choices**: the search or the mean;
- **six choices**: all of them.

Logistic regression and a small network (6 → 16 → 16 → choices), trained on the moving tiles of
the training split (notebook 3's table) to raise the expected score of the picture it picks - the
softmax over the choices times each picture's wPSNR, capped at 50 dB - and kept at the step that
scores best on a held-out tenth of the training sources (whole videos for the recordings). Scored
on the validation split as notebook 3. The one that scores best held out goes into the graph:
checked on real tiles, exported, timed.

Needs: `03_interpolate_tiles.ipynb` (its tables in `checkpoints/decide/`). Exports
`interpolate2x_tilesearch_decide_r4_s2.onnx` - into `dist/frame_gen_intra/` only if it beats the
search on the held-out tiles (section 3).

In [1]:
"""Settings, notebook 3's per-tile tables, and a tenth of the training sources held out."""

import copy
import pickle

import numpy as np
import torch

from helpers import clips, decide, models, runs, training, webexport
from helpers.training import DEVICE, to_tensor

SEED, CAP = 0, 50.0
KINDS = ("page", "desktop", "photo", "screen", "video")
CACHE = training.CHECKPOINT_DIR / "decide"
FEATURES = models.Decided.FEATURES
PUBLISHED = "±8"
training.seed_everything(SEED)
manifest = training.load_manifest()
tables = {split: pickle.loads((CACHE / f"{split}.pkl").read_bytes()) for split in ("train", "val")}
# notebook 3's reach: of the searches no slower in the browser than the published one, the best
# on the training split's moving tiles
REACH = decide.reach(tables["train"])
search = decide.search(REACH).to(DEVICE).eval()
print(f"the decision builds on the search {REACH}")


def flat(found, split):
    """One search's view of a table: its features, and the scores of its pictures beside the others'."""
    out = {key: value for key, value in found.items() if isinstance(value, np.ndarray)}
    out["features"] = found["features"][REACH]
    out["scores"] = {**found["scores"][REACH], **found["methods"], "published search": found["scores"][PUBLISHED]["search"]}
    listed = manifest["splits"][split]["sources"]
    out["kind"] = np.array([listed[s]["kind"] for s in found["source"]])
    out["group"] = np.array([listed[s].get("video", listed[s]["file"]) for s in found["source"]])
    return out


full = {split: flat(found, split) for split, found in tables.items()}


def subset(found, mask):
    out = {key: value[mask] for key, value in found.items() if isinstance(value, np.ndarray) and len(value) == len(mask)}
    out["scores"] = {name: value[mask] for name, value in found["scores"].items()}
    return out


groups = np.unique(full["train"]["group"])
held = set(np.random.default_rng(SEED).choice(groups, max(1, len(groups) // 10), replace=False))
in_held = np.isin(full["train"]["group"], list(held))
data = {"train": subset(full["train"], ~in_held), "held out": subset(full["train"], in_held), "val": full["val"]}
for name, found in data.items():
    print(f"{name:<9}{len(found['moving']):7} tiles, {found['moving'].mean():4.0%} moving - "
          + ", ".join(f"{k} {(found['kind'] == k).sum()}" for k in KINDS))

the decision builds on the search ±8
train      54968 tiles,  60% moving - page 10440, desktop 10208, photo 6496, screen 16993, video 10831
held out    5180 tiles,  58% moving - page 1624, desktop 928, photo 464, screen 1444, video 720
val        19374 tiles,  55% moving - page 3248, desktop 1392, photo 1740, screen 10652, video 2342


## 1. Learned decisions

Inputs: the six features, standardised on the training tiles. The loss is minus the expected
score: per tile, the softmax of the decision's logits times each choice's capped wPSNR (less their
mean over the choices, which changes nothing but the scale). Adam over the whole set; the step
kept is the one whose argmax scores best on the held-out tiles' moving ones.

In [2]:
def picked(found, choices, index):
    """(S,) the capped wPSNR of the picture `index` (S,) picks per tile among `choices`."""
    table = np.stack([np.minimum(found["scores"][c], CAP) for c in choices], 1)
    return table[np.arange(len(index)), index]


def fit(choices, kind_of_net, steps_max=3000, rate=1e-2):
    """(rule over a table, the module, (held-out score, weights, step), (mean, std) it standardises by)."""
    train = data["train"]
    use = train["moving"]
    x = train["features"][use]
    gains = np.stack([np.minimum(train["scores"][c], CAP) for c in choices], 1)[use]
    gains = gains - gains.mean(1, keepdims=True)
    mean, std = x.mean(0), x.std(0) + 1e-6
    torch.manual_seed(SEED)
    width = len(choices)
    net = torch.nn.Linear(len(FEATURES), width) if kind_of_net == "logistic" else torch.nn.Sequential(
        torch.nn.Linear(len(FEATURES), 16), torch.nn.ReLU(), torch.nn.Linear(16, 16), torch.nn.ReLU(), torch.nn.Linear(16, width))
    xt, gt = (torch.as_tensor(a, dtype=torch.float32) for a in ((x - mean) / std, gains))
    optimiser = torch.optim.Adam(net.parameters(), lr=rate)
    held_out = data["held out"]
    held_x = torch.as_tensor((held_out["features"] - mean) / std, dtype=torch.float32)
    best = (-np.inf, None, 0)
    for step in range(steps_max + 1):
        if step % 25 == 0:
            with torch.no_grad():
                score = picked(held_out, choices, net(held_x).argmax(1).numpy())[held_out["moving"]].mean()
            if score > best[0] + 1e-4:
                best = (score, copy.deepcopy(net.state_dict()), step)
        loss = -(torch.softmax(net(xt), 1) * gt).sum(1).mean()
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
    net.load_state_dict(best[1])

    def rule(found):
        with torch.no_grad():
            return net(torch.as_tensor((found["features"] - mean) / std, dtype=torch.float32)).argmax(1).numpy()
    return rule, net, best, (mean, std)


CHOICE_SETS = {"two": ("search", "mean"), "six": tuple(models.Decided.CHOICES)}
TRAINED = {}
for label, choices in CHOICE_SETS.items():
    for kind_of_net in ("logistic", "network"):
        name = f"{kind_of_net}, {label}"
        TRAINED[name] = (choices, *fit(choices, kind_of_net))
        print(f"{name:<20} held out: {TRAINED[name][3][0]:.3f} dB (step {TRAINED[name][3][2]})")
held_out = data["held out"]
print(f"{'(the search)':<20} held out: {np.minimum(held_out['scores']['search'], CAP)[held_out['moving']].mean():.3f} dB")
CHOSEN = max(TRAINED, key=lambda name: TRAINED[name][3][0])
print(f"to export: {CHOSEN}")

logistic, two        held out: 39.448 dB (step 0)
network, two         held out: 39.445 dB (step 775)
logistic, six        held out: 39.442 dB (step 400)
network, six         held out: 39.442 dB (step 25)
(the search)         held out: 39.442 dB
to export: logistic, two


## 2. Validation by kind

The validation split - never trained on nor used to choose. Absolute wPSNR on the moving tiles
(capped) per kind and over all; the gain over the search the decision builds on (notebook 3's
reach) on every moving tile with its
95% range over videos (clips for the drawn kinds); and the same over the changed and the unchanged
tiles, where a rule must not do harm. A learned rule picks on every tile, still or not.

In [3]:
found = data["val"]
rng = np.random.default_rng(SEED)
kind, group, moving, changed = found["kind"], found["group"], found["moving"], found["changed"]


def over(gain, mask, draws=2000):
    per = [gain[mask & (group == g)] for g in np.unique(group[mask])]
    means = [np.concatenate([per[i] for i in rng.integers(0, len(per), len(per))]).mean() for _ in range(draws)]
    return np.percentile(means, [2.5, 97.5])


base = np.minimum(found["scores"]["search"], CAP)
made = {"mean (the mock)": np.minimum(found["scores"]["mean"], CAP),
        f"tile search {PUBLISHED} (published)": np.minimum(found["scores"]["published search"], CAP),
        f"tile search + pick {PUBLISHED} (notebook 1)": np.minimum(found["scores"]["tile search + pick"], CAP),
        f"tile search {REACH}": base}
made.update({f"learned: {name}": picked(found, choices, rule(found)) for name, (choices, rule, *_ ) in TRAINED.items()})
print(f"{'moving tiles, ≤ 50 dB':<36}" + "".join(f"{k:>9}" for k in (*KINDS, "all")) + f"{'vs search':>11}{'95% over videos':>18}{'changed':>9}{'unchgd':>8}")
print(f"{'tiles':<36}" + "".join(f"{(moving & (kind == k)).sum():9}" for k in KINDS) + f"{moving.sum():9}")
for name, score in made.items():
    gain = score - base
    low, high = over(gain, moving) if name != f"tile search {REACH}" else (0.0, 0.0)
    print(f"{name:<36}" + "".join(f"{score[moving & (kind == k)].mean():9.2f}" for k in KINDS) + f"{score[moving].mean():9.2f}"
          f"{gain[moving].mean():+11.2f}     [{low:+.2f}, {high:+.2f}]{score[changed].mean():9.2f}{score[~changed].mean():8.2f}")

print(f"\nwhat each learned rule picks on the moving validation tiles")
for name, (choices, rule, *_ ) in TRAINED.items():
    index = rule(found)
    print(f"  {name:<18}" + ", ".join(f"{c} {np.mean(index[moving] == i):.0%}" for i, c in enumerate(choices)))

moving tiles, ≤ 50 dB                    page  desktop    photo   screen    video      all  vs search   95% over videos  changed  unchgd
tiles                                    1811      443     1176     5071     2191    10692
mean (the mock)                         29.03    38.22    34.86    43.20    35.75    38.15      -2.40     [-3.40, -1.58]    32.16   49.75
tile search ±8 (published)              35.81    38.20    42.35    43.70    36.69    40.55      +0.00     [+0.00, +0.00]    35.83   49.76
tile search + pick ±8 (notebook 1)      35.77    38.38    42.33    43.80    37.03    40.67      +0.11     [+0.04, +0.19]    36.01   49.76
tile search ±8                          35.81    38.20    42.35    43.70    36.69    40.55      +0.00     [+0.00, +0.00]    35.83   49.76
learned: logistic, two                  35.80    38.19    42.32    43.70    36.70    40.55      -0.01     [-0.02, +0.01]    35.82   49.76
learned: network, two                   35.80    38.22    42.35    43.69    36.70 

## 3. In the graph

`models.Decided` with the chosen decision inside: the six numbers from the search's costs,
standardised as in training, the decision, and per tile the winner's s and a, which scale the
search's motion and the mix of its two reads - the client's contract exactly, `previous` (N) and
`current` (N+2) in, `output` out. The table's features are `Decided.features` itself, so a decision
trained on the table is the graph's.

Checked on real tiles - the first clips of every kind of the validation split, rebuilt - against
the table's rule on the same frames: the picture the graph makes must be the candidate the rule
picks. Then exported twice, as notebook 1's graphs: the client's, with `N`, `H` and `W` symbolic,
and the benchmark page's at 36 tiles of 328x188 - a 1080p frame - each checked against torch on
real tiles and on noise and timed on the desktop CPU beside the published search. Into
`dist/frame_gen_intra/` only if the decision beats the search on the held-out tiles by more than
0.05 dB - extrapolation's rule for its search + correction (its notebook 3); otherwise into
`checkpoints/export/`, where a graph that adds nothing waits for a reason to be used.

In [4]:
choices, rule, net, best, (mean, std) = TRAINED[CHOSEN]
decided = models.Decided(search, copy.deepcopy(net), mean, std, choices).to(DEVICE).eval()
received = runs.received_every(manifest)
listed = manifest["splits"]["val"]["sources"]
agree, total, apart, real = 0, 0, 0.0, []
for kind_name in KINDS:
    for index in [i for i, s in enumerate(listed) if s["kind"] == kind_name][:2]:
        source = listed[index]
        stream = clips.rebuild(source, manifest)
        layout, size = clips.plan(stream.shape[2], stream.shape[1])
        tiles = [(n, t) for n in range(received, len(stream), received) if runs.valid(source, n, received)
                 for t in range(len(layout))][:128]
        if size != webexport.TILE_INPUT or not tiles:
            continue
        window = np.stack([stream[[n - received, n], layout[t][1]:layout[t][1] + size[0], layout[t][0]:layout[t][0] + size[1]]
                           for n, t in tiles])
        previous, current = to_tensor(window).unbind(1)
        with torch.no_grad():
            pictures, features = decided.candidates(previous, current)
            index = rule({"features": features.cpu().numpy()})
            take = decided.logits(previous, current).argmax(1).cpu().numpy()
            expected = torch.stack([pictures[choices[k]][i] for i, k in enumerate(index)])
            apart = max(apart, float((decided(previous, current) - expected).abs().max()))
        agree += (take == index).sum()
        total += len(tiles)
        real.append((previous[:8].cpu().numpy(), current[:8].cpu().numpy()))
print(f"{total} tiles of {len(real)} clips: the same choice on {agree}; "
      f"the picture at most {apart:.1e} from the candidate the table's rule picks")

920 tiles of 10 clips: the same choice on 920; the picture at most 0.0e+00 from the candidate the table's rule picks


In [5]:
FRAME_TILES = 36
family = "interpolate2x" if received == 2 else "interpolate"
settings = decide.SEARCHES[REACH]
name = f"{family}_tilesearch_decide_r{settings['radius']}_s{settings['stride']}" + ("" if settings["scale"] == 2 else f"_q{settings['scale']}")
label = f"tile search {REACH} px, a picture chosen per tile ({CHOSEN})"
step_h, step_w = webexport.TILE_STEP
real_previous, real_current = (np.concatenate(part)[:FRAME_TILES] for part in zip(*real))
module = decided.cpu().eval()
held_gain = best[0] - np.minimum(held_out["scores"]["search"], CAP)[held_out["moving"]].mean()
folder = webexport.MODELS_DIR if held_gain > 0.05 else training.CHECKPOINT_DIR / "export"
print(f"{CHOSEN} against the search on the held-out tiles: {held_gain:+.3f} dB -> {folder}")
for filename, batch, dynamic, tag in ((f"{name}.onnx", 1, True, label),
                                      (f"{name}_tile{step_w}x{step_h}_b{FRAME_TILES}.onnx", FRAME_TILES, False, f"{label} · 1080p frame")):
    info = webexport.export(module, filename, webexport.TILE_INPUT, label=tag, batch=batch, dynamic=dynamic, models_dir=folder)
    differences = [webexport.check_graph(info["path"], module, real_previous, real_current)]
    for n, h, w in ((FRAME_TILES, *webexport.TILE_INPUT), (4, *webexport.TILE_INPUT), (1, 32, 64))[:3 if dynamic else 1]:
        differences.append(webexport.check_graph(info["path"], module, *np.random.rand(2, n, 3, h, w).astype(np.float32)))
    ms = webexport.time_graph(info["path"], webexport.TILE_INPUT, batch=batch, seconds=2)
    webexport.annotate(info, cpu_ms=round(ms, 2), received_every=received)
    print(webexport.summary_line(info, ms))
    print(f"  as torch on real tiles and noise{' at 36 and 4 tiles of 328x188 and one of 64x32' if dynamic else ''}: "
          f"at most {max(differences):.1e} apart")
searched = webexport.time_graph(webexport.MODELS_DIR / f"{family}_tilesearch_r4_s2.onnx", webexport.TILE_INPUT, seconds=2)
print(f"\nthe published search ({PUBLISHED}) alone: {searched:.2f} ms a tile on the desktop CPU")
decided.to(DEVICE)

logistic, two against the search on the held-out tiles: +0.006 dB -> /home/henrik/projects/desktop-streamer-ai/src/frame_gen_intra/checkpoints/export
interpolate2x_tilesearch_decide_r4_s2.onnx            254.3 KB    2.97 ms   Absx1, Addx14, Castx11, Concatx8, Constantx84, ConstantOfShapex1, Convx4, Divx5, Equalx2, Expandx1, Gatherx9, Gemmx1, Greaterx1, GreaterOrEqualx4, GridSamplex2, Lessx1, Logx4, MatMulx3, Mulx16, Negx1, Rangex4, Reciprocalx4, ReduceMaxx3, ReduceMeanx1, ReduceMinx3, ReduceSumx3, Reshapex6, Shapex5, Slicex2, Softmaxx1, Sqrtx1, Subx8, Unsqueezex22, Wherex5   OUTSIDE BUDGET: Abs, Cast, ConstantOfShape, Div, Equal, Expand, Gather, Gemm, Greater, GreaterOrEqual, GridSample, Less, Log, MatMul, Mul, Neg, Range, Reciprocal, ReduceMax, ReduceMean, ReduceMin, ReduceSum, Reshape, Shape, Slice, Softmax, Sqrt, Sub, Unsqueeze, Where
  as torch on real tiles and noise at 36 and 4 tiles of 328x188 and one of 64x32: at most 9.5e-05 apart
interpolate2x_tilesearch_decide_r4_s2_tile320x

Decided(
  (search): TileSearch(
    (luma): Conv2d(3, 1, kernel_size=(2, 2), stride=(2, 2), bias=False)
    (moved): Conv2d(1, 81, kernel_size=(9, 9), stride=(2, 2), bias=False)
    (centre): Conv2d(1, 1, kernel_size=(9, 9), stride=(2, 2), bias=False)
  )
  (decision): Linear(in_features=6, out_features=2, bias=True)
)

## What it says

- **No learned decision beats the search.** Two choices or six, logistic regression or the small
  network: on the moving validation tiles −0.01 to +0.00 dB over the search (95% over videos
  within ±0.03). Held out, the best is +0.006 dB - logistic regression at its untrained start. Its
  graph is right (the same choice as the table's rule on all 920 real tiles checked, as torch to
  1e-4) and stays in `checkpoints/export/`.
- **What extrapolation's decision found is already in N+2.** There it gained where a motion stopped
  or was not a motion, which N−2 and N do not announce; between N and N+2 the search sees where
  the motion went, and is right on those tiles. What it gets wrong here is a change in one display
  frame, and that leaves nothing in its costs: the network on six choices learns to take the search
  on every moving tile, and logistic regression on two takes the mean on 68% of them and scores the
  same - where the costs say the search is unsure, the two pictures are about as good.
- **The one thing over the search on whole tiles is evidence at the pixel**: notebook 1's pick,
  +0.11 dB, which compares the search's two reads of every pixel - for 3.4 times the time.